# EGYPTIAN WORD DICTIONARY v10
### v10 changes over v9:
- **Word-level transliterations**: keys are now **space-split tokens** (not split by dash)
- `ḥm-kꜣ prꞽ-nb` → two entries: `hm-ka` and `pri-nb` (dash preserved as separator in key)
- This aligns with the Sign List where one sign can have a compound phonetic like `Hm-kA`
- Normalized key format: `[a-z]` + `-` only (dashes preserved, no other symbols)
- `MAX_STANDALONE_LEN` check removed — no longer splitting by dash, so all tokens are kept
- Covers both simple morphemes (`nb`) and compound sign readings (`hm-ka`, `pri-nb`)
- Fine-tuning ready: every form a word appears as in transliteration is captured
- All v9 normalization and cleaning logic kept intact

## CELL 0 — Install

In [ ]:
import subprocess, sys

def pip(*pkgs):
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs),
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'❌ pip error: {result.stderr[-400:]}')
    else:
        print(f'✅ Ready: {" ".join(pkgs)}')

pip('datasets', 'tqdm')
print('\n✅ All dependencies installed')

## CELL 1 — Normalization Core v10

In [ ]:
"""
egyptian_word_dictionary_v10
═════════════════════════════
v10 over v9:
  CHANGE 1: Word-level storage — split by SPACE only, not by dash.
             Each space-separated token is one dictionary entry.
             Dash is preserved in the normalized key as a morpheme separator.
             e.g.  'ḥm-kꜣ prꞽ-nb'  →  entries for 'hm-ka' and 'pri-nb'
             e.g.  'Hm-kA'           →  entry for 'hm-ka'
             e.g.  'nb'              →  entry for 'nb'

  CHANGE 2: normalize_token() introduced — like normalize_word() but preserves
             a single internal dash as morpheme boundary marker.
             'ḥm-kꜣ'  →  normalized: 'hm-ka',  original: 'ḥm-kꜣ'

  CHANGE 3: MAX_STANDALONE_LEN guard removed — no longer needed since we are
             not splitting compounds; the space boundary is the natural limit.

  CHANGE 4: 'word_length' is now the total phonetic char count (dashes excluded)
             for consistent filtering.

  All v9 fixes kept:
    - ɜ (U+025C) → 'a'
    - ⸮ (U+2E2E) stripped
    - Brackets stripped, content kept
    - Combining marks stripped
    - ≡ stripped
"""

import re
import unicodedata
from typing import Iterator

# ── Egyptian char map (normalized key only) ───────────────────────────────────
_EGYPTIAN_CHAR_MAP = [
    ('ꜣ','A'), ('Ꜣ','A'), ('ꜥ','a'), ('Ꜥ','a'),
    ('ḥ','H'), ('Ḥ','H'), ('ḫ','x'), ('Ḫ','x'), ('ẖ','X'),
    ('š','S'), ('Š','S'), ('ṯ','T'), ('Ṯ','T'), ('ḏ','D'), ('Ḏ','D'),
    ('ỉ','i'), ('Ị','i'), ('ı','i'), ('ꞽ','i'), ('Ꞽ','i'),
    ('â','a'), ('Â','a'), ('î','i'), ('Î','i'),
    ('ô','o'), ('û','u'), ('é','e'), ('è','e'), ('ê','e'),
    ('ɜ','a'), ('Ɜ','a'),  # v9 FIX 1: U+025C reversed open E → a
]

# Strip entirely (non-phonemic)
_STRIP_ENTIRELY_RE = re.compile(
    r'[=.,_?#!~^/\\]'
    r'|\u2261'              # ≡
    r'|\u2E2E'              # v9 FIX 2: ⸮ reversed question mark
    r'|[\u2039\u203a\u00ab\u00bb]'  # ‹›«»
)

_ARABIC_RE     = re.compile(r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]')

# For normalized key: strip everything EXCEPT letters and dash
_STRUCTURAL_NO_DASH_RE = re.compile(r'[,=._\[\]()/\\<>{}|!?#@~^]')
_NON_ALPHA_DASH_RE     = re.compile(r'[^a-zA-Z-]')  # keeps dash!
_MULTI_DASH_RE         = re.compile(r'-{2,}')         # collapse double-dashes
_EDGE_DASH_RE          = re.compile(r'^-+|-+$')        # strip leading/trailing dash


def _strip_garbage(t: str) -> str:
    """Remove Arabic chars + private/unassigned Unicode"""
    t = _ARABIC_RE.sub('', t)
    return ''.join(
        ch for ch in t
        if unicodedata.category(ch) not in ('Co', 'Cn', 'So', 'Cs')
    )


def clean_original(token: str) -> str:
    """
    Token as it appears in data — cleaned:
      - Strip brackets (Ps/Pe), keep content  →  dw(j)t → dwjt
      - Strip combining marks                 →  ḏ̣ → ḏ
      - Strip = , . _ ? # ! ~ ^ / ≡ ⸮ ‹›«»
      - Keep Egyptian chars ḫ ḥ š ṯ ḏ ꜣ ꜥ ꞽ as-is
      - Keep dash as morpheme separator        →  ḥm-kꜣ stays ḥm-kꜣ
    """
    if not isinstance(token, str):
        return ''
    t = _strip_garbage(token.strip())
    if not t:
        return ''

    # RULE 1: strip brackets, keep content
    t = ''.join(
        ch for ch in t
        if unicodedata.category(ch) not in ('Ps', 'Pe')
    )

    # RULE 2: strip combining marks
    t = ''.join(
        ch for ch in t
        if unicodedata.category(ch) != 'Mn'
    )

    # RULE 3: strip symbols (⸮ etc)
    t = _STRIP_ENTIRELY_RE.sub('', t)

    # Clean up dash edges/doubles
    t = _MULTI_DASH_RE.sub('-', t)
    t = _EDGE_DASH_RE.sub('', t)

    return t.strip()


def normalize_token(token: str) -> str:
    """
    v10 LOOKUP KEY — ASCII lowercase, dash preserved as morpheme separator.
    'ḥm-kꜣ'  →  'hm-ka'
    'prꞽ-nb' →  'pri-nb'
    'nb'      →  'nb'
    'dw(j)t'  →  'dwjt'
    y is NOT converted to i (consistent with v9)
    """
    t = clean_original(token)
    if not t:
        return ''
    # Apply Egyptian char map
    for src, dst in _EGYPTIAN_CHAR_MAP:
        t = t.replace(src, dst)
    # Strip structural chars (but NOT dash)
    t = _STRUCTURAL_NO_DASH_RE.sub('', t)
    t = t.lower()
    # Remove anything that's not [a-z] or dash
    t = _NON_ALPHA_DASH_RE.sub('', t)
    # Clean up dashes again after stripping
    t = _MULTI_DASH_RE.sub('-', t)
    t = _EDGE_DASH_RE.sub('', t)
    return t


def word_length_from_norm(norm: str) -> int:
    """Phonetic char count excluding dashes"""
    return len(norm.replace('-', ''))


def is_valid(norm: str) -> bool:
    return bool(norm and norm.replace('-', ''))


# ── Self-test ──────────────────────────────────────────────────────────────────
_TESTS = [
    # (token,                  exp_original,    exp_normalized, label)
    ('ḥm-kꜣ',                 'ḥm-kꜣ',         'hm-ka',        'Sign List D31: Hm-kA compound'),
    ('prꞽ-nb',                'prꞽ-nb',         'pri-nb',       'compound prꞽ-nb'),
    ('nb',                    'nb',              'nb',           'simple morpheme'),
    ('nHb-kAw',               'nHb-kAw',        'nhb-kaw',      'Sign D30: nHb-kAw'),
    ('Sni',                   'Sni',             'sni',          'simple with Egyptian S'),
    ('dw(j)t',                'dwjt',            'dwjt',         '() stripped, j kept'),
    ('dyɜ⟨r⟩',               'dyɜr',            'dyar',         'ɜ→a, ⟨⟩ stripped'),
    ('MAa,t-Raw',             'MAat-Raw',        'maat-raw',     'comma stripped, dash kept'),
    ('wDA',                   'wDA',             'wda',          'uppercase preserved → lower'),
    ('[nb]',                  'nb',              'nb',           '[] stripped'),
    ('anx-wDA-snb',           'anx-wDA-snb',    'anx-wda-snb',  'triple compound'),
    ('nty',                   'nty',             'nty',          'y stays y'),
    ('⸮ḫwtpl',               'ḫwtpl',           'xwtpl',        '⸮ stripped'),
    ('_',                     '',                '',             'garbage → empty'),
    ('Hm-kA',                 'Hm-kA',           'hm-ka',        'Sign List notation Hm-kA'),
]

print('🔬 v10 Self-test')
print('=' * 95)
print(f'{"Token":<22} {"Exp Orig":<18} {"Got Orig":<18} {"Exp Norm":<14} {"Got Norm":<14} Status')
print('-' * 95)
all_ok = True
for token, exp_o, exp_n, label in _TESTS:
    got_o = clean_original(token)
    got_n = normalize_token(token)
    ok = (got_o == exp_o) and (got_n == exp_n)
    if not ok: all_ok = False
    s = '✅' if ok else '❌'
    print(f'{repr(token):<22} {exp_o:<18} {got_o:<18} {exp_n:<14} {got_n:<14} {s}  [{label}]')
    if not ok:
        if got_o != exp_o: print(f'   original:   expected={exp_o!r} got={got_o!r}')
        if got_n != exp_n: print(f'   normalized: expected={exp_n!r} got={got_n!r}')
print()
print('✅ ALL PASSED' if all_ok else '❌ SOME FAILED')

## CELL 2 — Extract + Build Dictionary v10

In [ ]:
from typing import Iterator
from tqdm import tqdm


def extract_words(raw_sentence: str, source: str) -> Iterator[dict]:
    """
    v10: space split ONLY → tokens (each token = one transliteration word).
    Dash is preserved in the key as a morpheme separator.

    'ḥm-kꜣ prꞽ-nb'  →  2 entries:
        normalized='hm-ka',   original='ḥm-kꜣ'
        normalized='pri-nb',  original='prꞽ-nb'

    'nb'             →  1 entry:
        normalized='nb',      original='nb'

    Empty / all-punctuation tokens are skipped.
    """
    if not isinstance(raw_sentence, str):
        return
    for token in raw_sentence.split():
        orig = clean_original(token)
        norm = normalize_token(token)
        if not orig or not is_valid(norm):
            continue
        yield {
            'normalized' : norm,
            'original'   : orig,
            'source'     : source,
            'word_length': word_length_from_norm(norm),
        }


def build_dictionary(dataset_rows, column, source, word_dict, desc=''):
    rows_processed = words_added = duplicates = 0
    for row in tqdm(dataset_rows, desc=f'⏳ {desc}', unit='rows', dynamic_ncols=True):
        rows_processed += 1
        raw = row.get(column, '')
        for word_info in extract_words(raw, source):
            key = word_info['normalized']
            if key not in word_dict:
                word_dict[key] = {
                    'original'   : word_info['original'],
                    'source'     : word_info['source'],
                    'word_length': word_info['word_length'],
                }
                words_added += 1
            else:
                duplicates += 1
    return rows_processed, words_added, duplicates


# ── Demo ──────────────────────────────────────────────────────────────────────
print('🔬 extract_words() v10 demo')
print('=' * 65)
print(f'{"original":<25} {"normalized":<18} source')
print('-' * 60)

_DEMOS = [
    # sentence from Sign List D31 context
    ('ḥm-kꜣ prꞽ-nb',           'tla'),   # → 2 entries: hm-ka, pri-nb
    ('Hm-kA',                   'sign'),  # → hm-ka  (same key)
    ('nHb-kAw',                 'sign'),  # → nhb-kaw (D30)
    ('Sni|Sny|wS|Sn',           'sign'),  # pipes — each pipe-chunk is one token? NO: space-split only
    ('anx-wDA-snb',             'tla'),   # → anx-wda-snb  (one entry, triple compound)
    ('ḥtp nswt',                'bbaw'),  # → 2 entries: htp, nswt
    ('nty imy',                 'tla'),   # → 2 entries: nty, imy
    ('MAa,t-Raw',               'tla'),   # → maat-raw
    ('dw(j)t-nfr',              'bbaw'),  # → dwjt-nfr
    ('wDA',                     'tla'),   # → wda
    ('nb',                      'tla'),   # → nb
    # v9 compounds that were skipped before — now they enter as compound tokens
    ('anxwDAsnb',               'tla'),   # standalone no-dash → anxwdasnb (len=9, but kept!)
    ('⸮ḫwtpl',                 'bbaw'),  # ⸮ stripped → xwtpl
]
for sentence, src in _DEMOS:
    entries = list(extract_words(sentence, src))
    print(f'  Input: {repr(sentence)}')
    if entries:
        for e in entries:
            print(f'    {e["original"]:<25} {e["normalized"]:<18} {e["source"]}')
    else:
        print(f'    (skipped — empty after cleaning)')
    print()

print()
print('📌 NOTE: Sni|Sny|wS|Sn — pipes are NOT space separators.')
print('         The whole string is one token → normalized: sni|sny|ws|sn')
print('         If the Sign List col uses | as variant separator, handle separately.')

## CELL 2b — Handle Sign List Pipe-Separated Variants (Optional)

The Sign List column `B` (phonetic) uses `|` to separate variant readings, e.g.:
- `D3`: `Sni|Sny|wS|Sn`

This cell shows how to also ingest the Sign List itself as an extra source.

In [ ]:
# ── Optional: ingest Sign List with pipe-variant support ──────────────────────
# If you have the Sign List as a CSV/DataFrame with columns:
#   A = Gardiner code (e.g. D31)
#   B = phonetic variants pipe-separated (e.g. Sni|Sny|wS|Sn  or  Hm-kA)
#   C = type (Phonetic / Ideogram / Determinative)

def extract_sign_list_phonetics(phonetic_str: str, gardiner_code: str) -> Iterator[dict]:
    """
    Handle pipe-separated variant readings from the Sign List.
    'Sni|Sny|wS|Sn'  →  4 entries: sni, sny, ws, sn
    'Hm-kA'          →  1 entry:   hm-ka
    'nHb-kAw'        →  1 entry:   nhb-kaw
    """
    if not isinstance(phonetic_str, str) or not phonetic_str.strip():
        return
    variants = phonetic_str.split('|')
    for variant in variants:
        variant = variant.strip()
        if not variant:
            continue
        orig = clean_original(variant)
        norm = normalize_token(variant)
        if not orig or not is_valid(norm):
            continue
        yield {
            'normalized' : norm,
            'original'   : orig,
            'source'     : f'sign_list:{gardiner_code}',
            'word_length': word_length_from_norm(norm),
            'gardiner'   : gardiner_code,
        }


def build_dictionary_from_sign_list(sign_list_rows, word_dict, desc='Sign List'):
    """
    sign_list_rows: iterable of dicts with keys 'A' (Gardiner), 'B' (phonetic), 'C' (type)
    Only process Phonetic and Ideogram types (not Determinative).
    """
    rows_processed = words_added = duplicates = 0
    for row in tqdm(sign_list_rows, desc=f'⏳ {desc}', unit='rows', dynamic_ncols=True):
        rows_processed += 1
        phonetic = row.get('B', '') or row.get('phonetic', '')
        sign_type = row.get('C', '') or row.get('type', '')
        gardiner  = row.get('A', '') or row.get('gardiner', '')

        # Skip Determinatives (no phonetic value)
        if 'determinative' in str(sign_type).lower():
            continue

        for word_info in extract_sign_list_phonetics(phonetic, gardiner):
            key = word_info['normalized']
            if key not in word_dict:
                word_dict[key] = {
                    'original'   : word_info['original'],
                    'source'     : word_info['source'],
                    'word_length': word_info['word_length'],
                }
                words_added += 1
            else:
                duplicates += 1
    return rows_processed, words_added, duplicates


# ── Demo with Sign List samples from screenshot ───────────────────────────────
_SIGN_LIST_DEMO = [
    {'A': 'D2A',  'B': '',              'C': 'Determinative'},
    {'A': 'D3',   'B': 'Sni|Sny|wS|Sn','C': 'Phonetic'},
    {'A': 'D30',  'B': 'nHb-kAw',       'C': 'Ideogram'},
    {'A': 'D31',  'B': 'Hm-kA',         'C': 'Ideogram'},
    {'A': 'D300', 'B': '',              'C': 'Determinative'},
]

_demo_dict = {}
print('📋 Sign List extraction demo:')
print(f'{"Gardiner":<10} {"Input":<20} {"Original":<20} {"Normalized":<15} {"Type"}')
print('-' * 75)
for row in _SIGN_LIST_DEMO:
    for e in extract_sign_list_phonetics(row['B'], row['A']):
        print(f'{row["A"]:<10} {row["B"]:<20} {e["original"]:<20} {e["normalized"]:<15} {row["C"]}')
    if not row['B'] or 'determinative' in row['C'].lower():
        print(f'{row["A"]:<10} {"(skipped)":<20} — determinative or empty')

print()
print('✅ Pipe variants expanded, dashes preserved, determinatives skipped')

## CELL 3 — Load Datasets

In [ ]:
from datasets import load_dataset

# ── Dataset 1: BBAW Egyptian Corpus ──────────────────────────────────────────
print('⏳ Loading Dataset 1 — phiwi/bbaw_egyptian ...')
ds_bbaw = load_dataset('phiwi/bbaw_egyptian', split='train')
print(f'✅ BBAW loaded: {len(ds_bbaw):,} rows')
print(f'   Columns: {ds_bbaw.column_names}')
print(f'   Sample row [0] transcription: {repr(ds_bbaw[0].get("transcription", ""))[:100]}')
print(f'   Sample row [1] transcription: {repr(ds_bbaw[1].get("transcription", ""))[:100]}')

print()

# ── Dataset 2: TLA Earlier Egyptian Premium ───────────────────────────────────
print('⏳ Loading Dataset 2 — tla-Earlier_Egyptian_original-v18-premium ...')
ds_tla = load_dataset(
    'thesaurus-linguae-aegyptiae/tla-Earlier_Egyptian_original-v18-premium',
    split='train'
)
print(f'✅ TLA loaded: {len(ds_tla):,} rows')
print(f'   Columns: {ds_tla.column_names}')
print(f'   Sample row [0] transliteration: {repr(ds_tla[0].get("transliteration", ""))[:100]}')
print(f'   Sample row [1] transliteration: {repr(ds_tla[1].get("transliteration", ""))[:100]}')

## CELL 4 — Build Dictionary

In [ ]:
# ── Master dictionary ─────────────────────────────────────────────────────────
# Key   = normalized transliteration token, dash preserved (e.g. 'hm-ka', 'nb', 'anx-wda-snb')
# Value = {'original': str, 'source': str, 'word_length': int}
word_dictionary: dict[str, dict] = {}

print('=' * 65)
print('STAGE 1 — Processing BBAW Egyptian (transcription column)')
print('=' * 65)
rows1, added1, dups1 = build_dictionary(
    dataset_rows = ds_bbaw,
    column       = 'transcription',
    source       = 'bbaw',
    word_dict    = word_dictionary,
    desc         = 'BBAW',
)
print(f'\n   Rows processed : {rows1:>10,}')
print(f'   Words added    : {added1:>10,}')
print(f'   Duplicates skip: {dups1:>10,}')
print(f'   Dict size now  : {len(word_dictionary):>10,}')

print()
print('=' * 65)
print('STAGE 2 — Processing TLA Premium (transliteration column)')
print('=' * 65)
rows2, added2, dups2 = build_dictionary(
    dataset_rows = ds_tla,
    column       = 'transliteration',
    source       = 'tla',
    word_dict    = word_dictionary,
    desc         = 'TLA ',
)
print(f'\n   Rows processed : {rows2:>10,}')
print(f'   Words added    : {added2:>10,}')
print(f'   Duplicates skip: {dups2:>10,}')
print(f'   Dict size now  : {len(word_dictionary):>10,}')

print()
print('=' * 65)
print(f'🏛️  FINAL DICTIONARY: {len(word_dictionary):,} unique normalized word tokens')
print('    (each token = one space-separated transliteration unit)')
print('    (dashes preserved: hm-ka, anx-wda-snb, etc.)')
print('=' * 65)

## CELL 5 — Quality Checks

In [ ]:
import collections
import random

# ── 1. Key purity check: only [a-z] and dash ──────────────────────────────────
_DIRTY_KEY = re.compile(r'[^a-z-]')
dirty_entries = {k: v for k, v in word_dictionary.items() if _DIRTY_KEY.search(k)}

print('🔍 Key purity check (must be [a-z] and dash only):')
if dirty_entries:
    print(f'  ❌ {len(dirty_entries)} dirty keys found!')
    for k, v in list(dirty_entries.items())[:15]:
        print(f'     {repr(k):<30} ← original: {repr(v["original"])}')
else:
    print(f'  ✅ All {len(word_dictionary):,} keys are clean [a-z-] ASCII')

# ── 2. Dash stats ─────────────────────────────────────────────────────────────
compound_keys    = [k for k in word_dictionary if '-' in k]
simple_keys      = [k for k in word_dictionary if '-' not in k]
multi_dash_keys  = [k for k in word_dictionary if k.count('-') >= 2]
print(f'\n📊 Key structure:')
print(f'   Simple (no dash)     : {len(simple_keys):>8,}  ({len(simple_keys)/len(word_dictionary)*100:.1f}%)')
print(f'   Compound (1 dash)    : {len(compound_keys)-len(multi_dash_keys):>8,}  ({(len(compound_keys)-len(multi_dash_keys))/len(word_dictionary)*100:.1f}%)')
print(f'   Multi-compound (2+)  : {len(multi_dash_keys):>8,}  ({len(multi_dash_keys)/len(word_dictionary)*100:.1f}%)')
if compound_keys:
    print(f'   Sample compounds     : {", ".join(sorted(compound_keys)[:10])}')
if multi_dash_keys:
    print(f'   Sample multi-compound: {", ".join(sorted(multi_dash_keys)[:5])}')

# ── 3. Source distribution ────────────────────────────────────────────────────
print()
src_counts = collections.Counter(v['source'] for v in word_dictionary.values())
print('📊 Source distribution:')
total = len(word_dictionary)
for src, cnt in src_counts.most_common():
    print(f'   {src:<8}: {cnt:>8,}  ({cnt/total*100:.1f}%)')

# ── 4. Word length distribution (phonetic chars, dashes excluded) ─────────────
lengths  = [v['word_length'] for v in word_dictionary.values()]
len_dist = collections.Counter(lengths)
print(f'\n📊 Phonetic length distribution (dashes excluded):')
print(f'   Min    : {min(lengths)}')
print(f'   Max    : {max(lengths)}')
print(f'   Mean   : {sum(lengths)/len(lengths):.2f}')
print(f'   Median : {sorted(lengths)[len(lengths)//2]}')
print(f'   Top-15 length buckets:')
for length, cnt in sorted(len_dist.items())[:15]:
    bar = '█' * (cnt * 30 // max(len_dist.values()))
    print(f'   len={length:>2}: {cnt:>6,}  {bar}')

# ── 5. Key lookups: Sign List entries ─────────────────────────────────────────
print('\n📋 Sign List lookup test:')
print(f'{"Query":<22} {"Found?":<8} {"Original":<28} {"Source":<6} {"Len"}')
print('-' * 75)
_SIGN_CHECKS = [
    ('hm-ka',      'D31 Ideogram Hm-kA'),
    ('nhb-kaw',    'D30 Ideogram nHb-kAw'),
    ('sni',        'D3 Phonetic Sni'),
    ('sny',        'D3 Phonetic Sny'),
    ('ws',         'D3 Phonetic wS'),
    ('sn',         'D3 Phonetic Sn'),
    ('pri-nb',     'compound prꞽ-nb'),
    ('anx-wda-snb','triple compound'),
    ('nb',         'simple morpheme'),
    ('htp',        'simple morpheme'),
    ('maat',       'MAat'),
    ('maat-raw',   'MAat-Raw compound'),
]
for q, desc in _SIGN_CHECKS:
    entry = word_dictionary.get(q)
    if entry:
        print(f'{q:<22} {"✅ YES":<8} {entry["original"]:<28} {entry["source"]:<6} {entry["word_length"]}  ({desc})')
    else:
        print(f'{q:<22} ❌ NOT FOUND  ({desc})')

# ── 6. Random sample ─────────────────────────────────────────────────────────
print(f'\n📋 Random 20 entries (incl. compounds):')
print(f'{"Normalized Key":<25} {"Original":<28} {"Source":<6} {"Len"}')
print('-' * 70)
random.seed(42)
# Show mix of simple and compound
sample_simple   = random.sample(simple_keys,   min(10, len(simple_keys)))
sample_compound = random.sample(compound_keys, min(10, len(compound_keys)))
for k in sorted(sample_simple + sample_compound):
    v = word_dictionary[k]
    print(f'{k:<25} {v["original"]:<28} {v["source"]:<6} {v["word_length"]}')

## CELL 6 — Export CSV + JSON

In [ ]:
import os
import json
import pandas as pd


def _detect_output_dir() -> str:
    if os.path.exists('/kaggle/working'):
        return '/kaggle/working'
    try:
        import google.colab  # noqa
        return '/content'
    except ImportError:
        return '.'


OUT_DIR = _detect_output_dir()
print(f'📁 Output directory: {OUT_DIR}')

# ── Export 1: JSON ────────────────────────────────────────────────────────────
JSON_PATH = os.path.join(OUT_DIR, 'egyptian_word_dictionary_v10.json')
with open(JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(word_dictionary, f, ensure_ascii=False, indent=2)

size_mb = os.path.getsize(JSON_PATH) / 1024 / 1024
print(f'\n✅ JSON saved: {JSON_PATH}')
print(f'   Entries : {len(word_dictionary):,}')
print(f'   Size    : {size_mb:.2f} MB')

# ── Export 2: CSV ─────────────────────────────────────────────────────────────
CSV_PATH = os.path.join(OUT_DIR, 'egyptian_word_dictionary_v10.csv')
df_out = pd.DataFrame([
    {
        'normalized' : key,
        'original'   : val['original'],
        'source'     : val['source'],
        'word_length': val['word_length'],
        'is_compound': '-' in key,
        'morpheme_count': key.count('-') + 1,
    }
    for key, val in word_dictionary.items()
])
df_out = df_out.sort_values(['morpheme_count', 'word_length', 'normalized']).reset_index(drop=True)
df_out.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')

print(f'\n✅ CSV saved : {CSV_PATH}')
print(f'   Rows   : {len(df_out):,}')
print(f'   Columns: {list(df_out.columns)}')
print(f'   Simple entries    : {(~df_out["is_compound"]).sum():,}')
print(f'   Compound entries  : {df_out["is_compound"].sum():,}')

print()
print('=' * 65)
print('✅  PIPELINE COMPLETE — v10')
print('=' * 65)

## CELL 7 — Final Validation

In [ ]:
# ══════════════════════════════════════════════════════════════
# v10 FINAL VALIDATION
# ══════════════════════════════════════════════════════════════
print('🔬 v10 Final Validation')
print('=' * 70)

checks = [
    # (key,              exp_len, desc)
    # Simple morphemes — must still exist
    ('nb',              2,    '✅ simple morpheme nb'),
    ('htp',             3,    '✅ ḥtp'),
    ('anx',             3,    '✅ anx'),
    ('ra',              2,    '✅ Ra'),
    ('nty',             3,    '✅ y stays y'),
    ('wda',             3,    '✅ wDA uppercase'),
    ('maat',            4,    '✅ MAa,t'),
    ('dwjt',            4,    '✅ dw(j)t brackets'),
    # Compound tokens — v10 new
    ('anx-wda-snb',     9,    '✅ triple compound preserved'),
    ('maat-raw',        7,    '✅ MAat-Raw compound'),
    # v9 these were SKIPPED (standalone compounds without dash)
    # v10 they MUST exist (space-split only, no compound filter)
    ('anxwdasnb',       9,    '✅ v10: no-dash long token now kept'),
    # Old v9 morpheme lookups — must still work (they appear as space-isolated tokens)
    ('xrp',             3,    '✅ xrp sign morpheme'),
    ('hr',              2,    '✅ ḥr morpheme'),
    ('ib',              2,    '✅ ꞽb morpheme'),
]

print(f'{"Key":<22} {"Exp Len":>8}  {"Got Orig":<20} {"Got Len":>7}  Status')
print('-' * 75)
all_ok = True
for key, exp_len, desc in checks:
    entry = word_dictionary.get(key)
    ok = entry is not None and entry['word_length'] == exp_len
    got_orig = entry['original'] if entry else 'NOT FOUND'
    got_len  = entry['word_length'] if entry else '-'
    status   = '✅' if ok else f'❌ expected len={exp_len} got={got_len}'
    if not ok: all_ok = False
    print(f'{key:<22} {str(exp_len):>8}  {str(got_orig):<20} {str(got_len):>7}  {status}   {desc}')

print()
print(f'📊 Total entries: {len(word_dictionary):,}')
compound_count = sum(1 for k in word_dictionary if '-' in k)
print(f'   Simple   : {len(word_dictionary)-compound_count:,}')
print(f'   Compound : {compound_count:,}')
print('✅ ALL PASSED' if all_ok else '❌ SOME FAILED')